# DINOv2-large + projection 학습 (개 Re-ID) — Colab

pet-recognition-large 레시피 복제: **facebook/dinov2-large 백본은 얼리고, Linear(1024->512) projection만 ArcFace로 학습**.

- 학습 데이터: **Dogs of the World** (Kaggle `lextoumbourou/dogs-world`, **CC0 / Public Domain**). 313,688장 / 126,550 개체.
- 백본이 frozen이라 **이미지별 특징을 한 번만 뽑아 캐시**하면 head 학습은 초고속.
- 평가: 학습에 안 쓴 개체 일부를 held-out으로 떼어 케이스 단위 Recall@1/5/10.

### 준비 (한 번만)
1. Kaggle에서 dogs-world 데이터셋 zip을 받는다 (`archive.zip`, 몇 GB).
2. Google Drive에 업로드한다. 예: `MyDrive/dogsworld/archive.zip`
3. 아래 CONFIG의 `DATA_ZIP` 경로를 맞춘다.

> zip 안 구조: `archive/images/<hash>.png|jpg`, `archive/metadata/<hash>.json` (json에 identities[].identity, path), `archive/identities.json`

In [ ]:
# --- 1. 설치 & 임포트 ---
!pip -q install -U transformers
import os, json, glob, time, random, zipfile, collections
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEV, '|', torch.cuda.get_device_name(0) if DEV == 'cuda' else '')

In [ ]:
# --- 2. CONFIG (여기만 만지면 됨) ---
from google.colab import drive; drive.mount('/content/drive')

DATA_ZIP = '/content/drive/MyDrive/dogsworld/archive.zip'   # Drive에 올린 zip
WORK     = '/content/archive'                               # zip 풀 곳 (코랩 로컬 = 빠름)
OUT_DIR  = '/content/drive/MyDrive/dogsworld/out'           # 체크포인트·특징 캐시 (Drive)

MIN_PHOTOS  = 3       # 학습 개체는 사진 이 장수 이상
N_TRAIN_IDS = 16000   # 학습 개체 수 (pet-recognition-large 16,469). T4면 12k~20k
N_EVAL_IDS  = 500     # held-out 평가 개체 수 (사진 2장 이상 중)
AUG_VIEWS   = 1       # 이미지당 캐시 특징 수 (1=증강없음/빠름)

PROJ_DIM   = 512
EPOCHS     = 60
LR         = 1e-3
BATCH_HEAD = 1024
ARC_MARGIN = 0.3
ARC_SCALE  = 32.0
EVAL_EVERY = 5
FEAT_BATCH = 48      # DINOv2 특징추출 배치 (T4 15GB)
SEED       = 0

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

In [ ]:
# --- 3. zip 풀기 (코랩 로컬로) — 이미 있으면 건너뜀 ---
if not os.path.isdir(os.path.join(WORK, 'metadata')):
    assert os.path.exists(DATA_ZIP), 'zip 없음: ' + DATA_ZIP
    t = time.time(); print('unzip 중...', DATA_ZIP)
    with zipfile.ZipFile(DATA_ZIP) as z: z.extractall('/content')
    if not os.path.isdir(WORK):
        cand = [p for p in glob.glob('/content/*') if os.path.isdir(p) and os.path.isdir(p + '/metadata')]
        assert cand, 'archive/metadata 구조를 못 찾음'
        os.rename(cand[0], WORK)
    print('done', round(time.time() - t), 's')
print('images  :', len(os.listdir(WORK + '/images')))
print('metadata:', len(os.listdir(WORK + '/metadata')))

In [ ]:
# --- 4. 메타데이터 -> 개체별 이미지 목록 (Drive에 캐시) ---
cache_map = os.path.join(OUT_DIR, 'id_to_paths.json')
if os.path.exists(cache_map):
    id2paths = json.load(open(cache_map)); print('캐시 로드')
else:
    id2paths = collections.defaultdict(list); t = time.time()
    for i, fn in enumerate(os.listdir(WORK + '/metadata')):
        d = json.load(open(WORK + '/metadata/' + fn))
        img = os.path.join(WORK, d['path'])
        if not os.path.exists(img):
            b = os.path.splitext(img)[0]
            for e in ('.png', '.jpg', '.jpeg', '.webp'):
                if os.path.exists(b + e): img = b + e; break
        for ent in d.get('identities', []):
            id2paths[ent['identity']].append(img)
        if i % 60000 == 0: print(' ', i)
    id2paths = dict(id2paths)
    json.dump(id2paths, open(cache_map, 'w'))
    print('빌드', round(time.time() - t), 's')
dist = collections.Counter(len(v) for v in id2paths.values())
print('고유 개체:', len(id2paths))
for k in sorted(dist):
    if k <= 6 or k % 5 == 0: print('  %d장: %d' % (k, dist[k]))
for th in (2, 3, 4):
    print('  >=%d장: %d' % (th, sum(v for k, v in dist.items() if k >= th)))

In [ ]:
# --- 5. 학습/평가 개체 분리 ---
ge2 = sorted(k for k, v in id2paths.items() if len(v) >= 2)
random.Random(SEED).shuffle(ge2)
eval_ids = ge2[:N_EVAL_IDS]; eval_set = set(eval_ids)
train_pool = [k for k in id2paths if len(id2paths[k]) >= MIN_PHOTOS and k not in eval_set]
random.Random(SEED + 1).shuffle(train_pool)
train_ids = train_pool[:N_TRAIN_IDS]
pid_of = {k: i for i, k in enumerate(train_ids)}
train_items = [(p, pid_of[k]) for k in train_ids for p in id2paths[k]]
eval_q, eval_g = [], []
for k in eval_ids:
    ps = id2paths[k]; nq = max(1, len(ps) // 2)
    eval_g += [(p, k) for p in ps[:-nq]]; eval_q += [(p, k) for p in ps[-nq:]]
print('train: %d 개체 / %d 장' % (len(train_ids), len(train_items)))
print('eval : %d 개체 | query %d / gallery %d' % (len(eval_ids), len(eval_q), len(eval_g)))

In [ ]:
# --- 6. DINOv2-large (frozen) & 특징추출 ---
from transformers import AutoModel
backbone = AutoModel.from_pretrained('facebook/dinov2-large').eval().to(DEV)
for p in backbone.parameters(): p.requires_grad_(False)

class ImgDS(Dataset):
    def __init__(s, paths, tf): s.paths, s.tf = paths, tf
    def __len__(s): return len(s.paths)
    def __getitem__(s, i):
        try: im = Image.open(s.paths[i]).convert('RGB')
        except Exception: im = Image.new('RGB', (224, 224), (114, 114, 114))
        return s.tf(im)

aug_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.7, 1.0)), T.RandomHorizontalFlip(),
                    T.ColorJitter(0.2, 0.2, 0.2), T.ToTensor(), T.Normalize(MEAN, STD)])
plain_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])

@torch.no_grad()
def feats(paths, tf, bs=FEAT_BATCH):
    dl = DataLoader(ImgDS(paths, tf), batch_size=bs, num_workers=2, pin_memory=True)
    out = []; t = time.time()
    for j, x in enumerate(dl):
        out.append(backbone(pixel_values=x.to(DEV)).pooler_output.half().cpu())
        if j % 200 == 0: print('  %d/%d  %ds' % (j * bs, len(paths), round(time.time() - t)), flush=True)
    return torch.cat(out)

In [ ]:
# --- 7. 학습 특징 캐시 (Drive) — 있으면 건너뜀 ---
tag = '%did_%dmin_%dv_s%d' % (N_TRAIN_IDS, MIN_PHOTOS, AUG_VIEWS, SEED)
feat_path = os.path.join(OUT_DIR, 'trainfeat_%s.pt' % tag)
if os.path.exists(feat_path):
    blob = torch.load(feat_path); F_train, y_train = blob['feat'], blob['label']
    print('특징 캐시 로드', tuple(F_train.shape))
else:
    paths = [p for p, _ in train_items]; labs = torch.tensor([y for _, y in train_items])
    banks = [feats(paths, plain_tf)]; ys = [labs]
    for v in range(AUG_VIEWS - 1):
        print('aug view', v + 1); banks.append(feats(paths, aug_tf)); ys.append(labs)
    F_train = torch.cat(banks); y_train = torch.cat(ys)
    torch.save({'feat': F_train, 'label': y_train}, feat_path)
    print('저장', feat_path, tuple(F_train.shape))

In [ ]:
# --- 8. 평가 특징 캐시 ---
ev_path = os.path.join(OUT_DIR, 'evalfeat_%d_s%d.pt' % (N_EVAL_IDS, SEED))
if os.path.exists(ev_path):
    eb = torch.load(ev_path); Fq, IQ, Fg, IG = eb['Fq'], eb['IQ'], eb['Fg'], eb['IG']
    print('평가 특징 캐시 로드')
else:
    Fq = feats([p for p, _ in eval_q], plain_tf); IQ = [k for _, k in eval_q]
    Fg = feats([p for p, _ in eval_g], plain_tf); IG = [k for _, k in eval_g]
    torch.save({'Fq': Fq, 'IQ': IQ, 'Fg': Fg, 'IG': IG}, ev_path)
    print('저장', ev_path)

In [ ]:
# --- 9. ArcFace + 케이스단위 Recall ---
class ArcFace(nn.Module):
    def __init__(s, dim, n, sc=32.0, m=0.3):
        super().__init__(); s.W = nn.Parameter(torch.empty(n, dim)); nn.init.xavier_uniform_(s.W); s.sc, s.m = sc, m
    def forward(s, f, y):
        cos = f @ F.normalize(s.W, dim=1).t()
        th = torch.acos(cos.clamp(-1 + 1e-7, 1 - 1e-7))
        oh = F.one_hot(y, cos.size(1)).float()
        return F.cross_entropy(s.sc * (oh * torch.cos(th + s.m) + (1 - oh) * cos), y)

@torch.no_grad()
def case_recall(proj):
    proj.eval()
    zq = F.normalize(proj(Fq.float().to(DEV)), dim=1).cpu().numpy()
    zg = F.normalize(proj(Fg.float().to(DEV)), dim=1).cpu().numpy()
    IQa, IGa = np.array(IQ), np.array(IG)
    cps = sorted(set(IQ))
    ce = np.stack([(lambda v: v / (np.linalg.norm(v) + 1e-12))(zq[IQa == c].mean(0)) for c in cps])
    gids = sorted(set(IG)); gi = {g: i for i, g in enumerate(gids)}
    sim = ce @ zg.T
    M = np.full((len(cps), len(gids)), -1.0, np.float32)
    for j, g in enumerate(IGa): M[:, gi[g]] = np.maximum(M[:, gi[g]], sim[:, j])
    return {k: sum(1 for i, c in enumerate(cps) if c in [gids[j] for j in np.argsort(-M[i])][:k]) / len(cps) for k in (1, 5, 10)}

In [ ]:
# --- 10. projection 학습 ---
n_cls = len(train_ids)
proj = nn.Linear(F_train.size(1), PROJ_DIM, bias=False).to(DEV)
arc = ArcFace(PROJ_DIM, n_cls, sc=ARC_SCALE, m=ARC_MARGIN).to(DEV)
opt = torch.optim.AdamW([*proj.parameters(), *arc.parameters()], lr=LR, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
Fd = F_train.float().to(DEV); yd = y_train.to(DEV); N = Fd.size(0)

class _Id(nn.Module):
    def forward(s, x): return x
print('zero-shot (projection 없이 raw DINOv2-large):', {k: round(v, 3) for k, v in case_recall(_Id()).items()})

best = {'r1': -1, 'ep': -1}
for ep in range(1, EPOCHS + 1):
    proj.train(); perm = torch.randperm(N, device=DEV); tot = 0.0
    for i in range(0, N, BATCH_HEAD):
        idx = perm[i:i + BATCH_HEAD]
        loss = arc(F.normalize(proj(Fd[idx]), dim=1), yd[idx])
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(idx)
    sch.step()
    if ep % EVAL_EVERY == 0 or ep == EPOCHS:
        r = case_recall(proj)
        print('[ep {:3d}] loss {:.3f} lr {:.2e} | held-out  R@1 {:.1%}  R@5 {:.1%}  R@10 {:.1%}'.format(ep, tot / N, opt.param_groups[0]['lr'], r[1], r[5], r[10]))
        if r[1] > best['r1']:
            best.update(r1=r[1], ep=ep)
            torch.save({'proj': proj.state_dict(), 'backbone': 'facebook/dinov2-large',
                        'proj_dim': PROJ_DIM, 'r1': r[1], 'epoch': ep, 'n_train_ids': n_cls},
                       os.path.join(OUT_DIR, 'dinov2_proj_dog.pth'))
            print('    -> best 저장  R@1 {:.1%}'.format(r[1]))
print('\n최고 held-out R@1 {:.1%} @ epoch {}  ->  {}/dinov2_proj_dog.pth'.format(best['r1'], best['ep'], OUT_DIR))

In [ ]:
# --- 11. 추론용 로더 (다른 데이터/서비스에 쓸 때) ---
class DinoProj(nn.Module):
    def __init__(s, ckpt):
        super().__init__()
        b = AutoModel.from_pretrained(ckpt['backbone']).eval()
        for p in b.parameters(): p.requires_grad_(False)
        s.b = b
        s.p = nn.Linear(1024, ckpt['proj_dim'], bias=False); s.p.load_state_dict(ckpt['proj'])
    @torch.no_grad()
    def forward(s, x):  # x: [N,3,224,224], ImageNet 정규화
        return F.normalize(s.p(s.b(pixel_values=x).pooler_output), dim=1)

ck = torch.load(os.path.join(OUT_DIR, 'dinov2_proj_dog.pth'))
model = DinoProj(ck).eval().to(DEV)
print('로드 OK. 512-d L2정규화 임베딩 | 전처리 Resize(224,224)+ImageNet mean/std | 매칭 코사인')

## 메모

- **데이터셋 라이선스: CC0 (Public Domain)** — 상업 배포 포함 자유, 출처 표기 의무 없음.
- 이 데이터셋은 pet-recognition-large가 개 projection 학습에 쓴 것과 사실상 동일 -> 목표는 **재현/소유**(네이티브 torch, 재학습 가능)지 성능 초과가 아님.
- held-out 평가는 **같은 도메인**(Dogs-World 미학습 개체)이라 낙관적일 수 있음. 진짜 목표 도메인 평가는 shelter_hard_dogs(한국 보호소)를 Drive에 올려 8번 셀을 그쪽으로 바꿔 실행.
- T4에서 **특징추출이 병목**. `N_TRAIN_IDS` 낮추거나 `AUG_VIEWS=1`. 특징은 Drive에 캐시되니 재실행은 학습만 (수 분).
- 셀 10에서 **raw DINOv2-large zero-shot**을 같은 held-out으로 함께 출력 — 그 수치를 넘겨야 projection이 의미.